# HypeCheck: can tweet hype predict next-day memecoin moves?

> **Archived experiment.** The [current submission](https://github.com/adeviwon/hypecheck) uses the 13-coin time split and the main HypeCheck notebook.

**Veritas AI International High School Hackathon 2026**

**Track:** AI + Finance (built on suggested project 4.2.4, *News headlines and the stock market*)
**Team:** Nathaniel Wong (model) · Aaden Yick (data) · Alastair Chu (demo and write-up) — Malvern College Hong Kong

**One-sentence pitch:** HypeCheck reads a memecoin tweet and tells a beginner investor whether the hype in it carries any predictive signal about tomorrow's price — and reports honestly that it does not.

**How to run:** Runtime → Run all. It takes about two minutes, needs no API keys, and downloads two public inputs which it verifies by SHA-256 before training on them.

*This project is a prototype and is not financial advice.*

## 1. Load the data

Two public sources:

| # | Source | What we take | Licence |
|---|---|---|---|
| 1 | [`StephanAkkerman/financial-tweets-crypto`](https://huggingface.co/datasets/StephanAkkerman/financial-tweets-crypto) | 104,379 X posts with text, ticker tags and a timestamp for when the archive first saw them | MIT |
| 2 | [Binance spot daily bars](https://github.com/binance/binance-spot-api-docs) | open, high, low, close and volume for nine memecoins (USDT pairs) | Public market data |

The tweet file is fetched from Hugging Face at a pinned revision. The price files are fetched from our [repository](https://github.com/adeviwon/hypecheck) rather than a live Binance call, because Binance blocks some regions and cloud IP ranges — a submission that only runs in certain countries is not a submission. Every file is checked against a SHA-256 digest before anything is trained, so bad bytes stop the run instead of quietly changing a number.

In [1]:
# 1 · Fetch both inputs and verify them.
import hashlib
from pathlib import Path

import requests

ROOT = Path.cwd()
DATA = ROOT / "data"
DATA.mkdir(exist_ok=True)
RAW = "https://raw.githubusercontent.com/adeviwon/hypecheck/main/data"
FILES = {
    "DOGE-prices.csv": "afc23a84c958279129a02a5e556db41218bcfaae2a403ff100f88d634f44f2c2",
    "SHIB-prices.csv": "6f65ac812bbcbde740ecf9b34120147fe786999ec67df8631508027a5afb719b",
    "PEPE-prices.csv": "024ecb4a9da72818a06d4b86e7d8f87c3c51359a97dab27962a73d5bd609702f",
    "BONK-prices.csv": "18be2016111fa5fc051ba750ca21fa2c7a883ccdc80986f03ae23069a38fee02",
    "FLOKI-prices.csv": "1348af136631c9b4167ea5656aeae5ff1ea567300ae8492a5e92fb81b314d82e",
    "WIF-prices.csv": "ff1c2b92ed89d29e8c2de049c3f74693f0ff3fe9d2ad9f168e1291d7ec93b89a",
    "ACT-prices.csv": "5c23b8e8b1bc5d325a87a38872751bcb1b7663211a4ef7907bacd2c04e531634",
    "PNUT-prices.csv": "9ce1baa613d374a66aca3fdcffde4847a95e7c261f944a116753381472a74286",
    "BOME-prices.csv": "0fc0fb1d1f16588c8459257f99677495f8ac609bfa3597438c4904f56976d278",
    "hype-lexicon.csv": "598d46cb09ddba8571680b36de37a95b9e02432d82c638f65597a327e42a5895",
}
# The tweet file is 20 MB of source data, so we take it from Hugging Face at a pinned revision
# rather than republishing a second copy in our own repository.
HF_REVISION = "c4b17a94a667af3da1c9ed5144f998e3a4f29966"
POSTS_URL = ("https://huggingface.co/datasets/StephanAkkerman/financial-tweets-crypto/resolve/"
             + HF_REVISION + "/data/train.parquet")
POSTS_SHA = "81b2f7563a44c1a103e9468a83d52319dfeae0fc616422b4a4df90d8d684b960"

def sha256(path):
    with open(path, "rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

sources = [("posts.parquet", POSTS_URL, POSTS_SHA)] + [(n, f"{RAW}/{n}", s) for n, s in FILES.items()]

for name, url, expected in sources:
    dest = DATA / name
    if dest.exists() and sha256(dest) == expected:
        print(f"  ok  {name:20s} {dest.stat().st_size / 1e6:8.2f} MB  already present, digest matches")
        continue
    response = requests.get(url, timeout=(10, 300))
    response.raise_for_status()
    dest.write_bytes(response.content)
    if sha256(dest) != expected:
        raise ValueError(f"{name} does not match its pinned SHA-256; refusing to continue")
    print(f"  ok  {name:20s} {dest.stat().st_size / 1e6:8.2f} MB  downloaded and verified")

print("\nEvery input verified by SHA-256. Nothing here is invented.")

  ok  posts.parquet           20.74 MB  downloaded and verified
  ok  DOGE-prices.csv          0.07 MB  downloaded and verified
  ok  SHIB-prices.csv          0.08 MB  downloaded and verified
  ok  PEPE-prices.csv          0.08 MB  downloaded and verified
  ok  BONK-prices.csv          0.08 MB  downloaded and verified
  ok  FLOKI-prices.csv         0.09 MB  downloaded and verified
  ok  WIF-prices.csv           0.06 MB  downloaded and verified
  ok  ACT-prices.csv           0.05 MB  downloaded and verified
  ok  PNUT-prices.csv          0.05 MB  downloaded and verified
  ok  BOME-prices.csv          0.07 MB  downloaded and verified
  ok  hype-lexicon.csv         0.01 MB  downloaded and verified

Every input verified by SHA-256. Nothing here is invented.


## 2. Look at the data before training on it

We join each tweet to the next day's price move of the coin it tags.

**The rules, stated up front** (they matter, so we are not hiding them):

- **Original posts only.** Retweets and quote tweets are dropped, so we are not counting the same opinion twice.
- **When information was available.** A post is only usable from `max(written_at, first_seen_by_archive)`. We take the *later* of the two, which is the conservative choice, and only ever use a post to predict a day that starts *after* it was available.
- **The target.** Did the coin's close finish above its open on the next UTC day? A flat day counts as *not up*.
- **The coin must have traded.** A tweet is dropped if there is no daily bar for the day it would predict.

We split the coins into two groups. The **large-cap group** is the five best-known memecoins. The **pump.fun-era group** is four Solana memecoins from the launchpad era of late 2023 onwards. We train only on the large-cap group and use the other as an unseen test set in section 7.

In [2]:
# 2 · Build the labelled tweet panel.
import numpy as np
import pandas as pd

LARGE = ["DOGE", "SHIB", "PEPE", "BONK", "FLOKI"]
NEW   = ["WIF", "ACT", "PNUT", "BOME"]

posts = pd.read_parquet("data/posts.parquet")
print(f"Source posts: {len(posts):,}   columns include: {list(posts.columns)[:6]} ...")

posts = posts.loc[posts.tweet_type.eq("tweet")]
posts = posts.loc[posts.text.fillna("").astype(str).str.len().gt(0)]
posts = posts.drop_duplicates(subset=["tweet_id"])
written = pd.to_datetime(posts.created_at, utc=True, errors="coerce")
seen    = pd.to_datetime(posts.first_posted_at, utc=True, errors="coerce")
posts = posts.assign(available_at=pd.concat([written, seen], axis=1).max(axis=1))
posts = posts.dropna(subset=["available_at"])
print(f"Original posts with usable text and a timestamp: {len(posts):,}")

def build_panel(coins):
    frames = []
    for coin in coins:
        prices = pd.read_csv(f"data/{coin}-prices.csv", parse_dates=["date"])
        prices = prices.assign(day=prices.date.dt.date, up=(prices.close > prices.open).astype(int))
        tagged = posts.tickers.apply(lambda x: x is not None and any(str(v).upper() == coin for v in x))
        sub = posts.loc[tagged].copy()
        if not len(sub):
            continue
        # the post predicts the day AFTER it became available
        sub["trading_day"] = (sub.available_at.dt.normalize() + pd.Timedelta(days=1)).dt.date
        sub["coin"] = coin
        joined = sub.merge(prices[["day", "up"]], left_on="trading_day", right_on="day", how="inner")
        frames.append(joined[["tweet_id", "coin", "available_at", "trading_day", "text", "up"]])
    return pd.concat(frames, ignore_index=True)

big = build_panel(LARGE)
new = build_panel(NEW)

for label, frame, coins in [("large-cap", big, LARGE), ("pump.fun-era", new, NEW)]:
    print(f"\n{label} group: {len(frame):,} labelled tweets, next-day up rate {frame.up.mean():.1%}")
    for coin in coins:
        c = frame.loc[frame.coin.eq(coin)]
        if len(c):
            print(f"    {coin:6s} {len(c):5,d} tweets   up rate {c.up.mean():5.1%}   "
                  f"{str(c.available_at.min())[:10]} to {str(c.available_at.max())[:10]}")

Source posts: 104,379   columns include: ['tweet_id', 'created_at', 'source', 'categories', 'url', 'user_name'] ...
Original posts with usable text and a timestamp: 78,381

large-cap group: 1,264 labelled tweets, next-day up rate 48.1%
    DOGE     669 tweets   up rate 45.7%   2024-02-12 to 2026-10-16
    SHIB      46 tweets   up rate 50.0%   2024-04-07 to 2026-10-19
    PEPE     254 tweets   up rate 51.6%   2024-02-12 to 2026-10-11
    BONK     143 tweets   up rate 49.7%   2024-02-17 to 2026-10-11
    FLOKI    152 tweets   up rate 50.7%   2024-03-22 to 2026-10-11

pump.fun-era group: 274 labelled tweets, next-day up rate 45.3%
    WIF      164 tweets   up rate 43.3%   2024-03-31 to 2026-10-02
    ACT       61 tweets   up rate 44.3%   2024-11-11 to 2025-02-25
    PNUT      33 tweets   up rate 63.6%   2024-11-11 to 2026-07-08
    BOME      16 tweets   up rate 31.2%   2024-03-22 to 2026-08-21


Two things to notice immediately.

**The up rate is close to a coin flip** in both groups. That is the first hint that this will be a hard problem: if roughly half of all days go up, a model that always says "up" is already half right, so any accuracy figure has to be read against that base rate rather than against zero.

**The sample is small and lopsided.** One coin (DOGE) supplies most of the large-cap rows, and the pump.fun-era group is under 300 tweets in total. We flag this now because it bounds everything that follows.

In [3]:
# 2b · Read some of the tweets ourselves, as the brief's own advice suggests.
sample = big.sample(6, random_state=7)
for _, row in sample.iterrows():
    text = " ".join(str(row.text).split())[:150]
    print(f"[{str(row.available_at)[:10]}] next day {'UP  ' if row.up else 'down'}  {row.coin:5s}  {text}")

print(f"\nText length: median {big.text.str.len().median():.0f} characters, "
      f"longest {big.text.str.len().max():,}")

[2025-03-31] next day UP    FLOKI  $FLOKI is building a bullish divergence, right on top of the major support level. Time to turn the bear market ceiling into the bull market floor? Let
[2024-10-28] next day UP    DOGE   $DOGE giga sending, $FLOKI to follow soon Source: my quantsack
[2024-11-11] next day UP    DOGE   If I can close out this $DOGE scalp for a quick 50 grand before I go to bed, I’d say that’s a good Sunday afternoon’s work. 😀
[2025-04-22] next day UP    FLOKI  $FLOKI making a significant break out today and now showing relative strength. Pretty big upside potential.
[2024-10-19] next day down  DOGE   $DOGE/BTC My DOGE calls are for the history books, but hey let's ignore actual greatness. Sharing this pair so your guru can copy pasta from a ghost b
[2025-05-14] next day down  DOGE   I understood the memecoin thing when it was just $DOGE back in the day. I don't understand the 500 different memecoins thing now. Was more special whe

Text length: median 87 characters, long

Most of these posts are technical-analysis commentary rather than shilling: they talk about support, breakouts and cycle positions. That is an important observation about *this* dataset, and it comes back in section 9 — the vocabulary people expect in memecoin hype ("to the moon", "100x") turns out to be almost entirely absent from it.

## 3. Split into training and test sets

We hold back 20% of the tweets and never train on them, exactly as the brief asks. `stratify` keeps the same share of "up" days in both halves. `random_state=42` makes the split reproducible.

In [4]:
# 3 · 80/20 split.
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    big.text, big.up, test_size=0.2, stratify=big.up, random_state=42
)
print(f"Training: {len(X_train):,} tweets   Test: {len(X_test):,} tweets")
print(f"Up rate — training {y_train.mean():.1%}, test {y_test.mean():.1%}")

Training: 1,011 tweets   Test: 253 tweets
Up rate — training 48.1%, test 48.2%


## 4. Baseline: always guess the majority class

The simplest possible rule. Any model we build has to beat it, and it sets the number we have to read every later accuracy against.

In [5]:
# 4 · The baseline, and the scoring function we use from here on.
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score

def report(name, y_true, y_pred):
    print(f"{name}")
    print(f"  accuracy  {accuracy_score(y_true, y_pred):.1%}")
    print(f"  precision {precision_score(y_true, y_pred, zero_division=0):.1%}  (of 'up' calls, how many were really up)")
    print(f"  recall    {recall_score(y_true, y_pred, zero_division=0):.1%}  (of real up days, how many we caught)")

majority = int(y_train.mean() > 0.5)
report(f"Baseline: always '{'up' if majority else 'down'}'", y_test, np.full(len(y_test), majority))

Baseline: always 'down'
  accuracy  51.8%
  precision 0.0%  (of 'up' calls, how many were really up)
  recall    0.0%  (of real up days, how many we caught)


The baseline scores **51.8%** while understanding nothing at all. Everything below has to be read against that number, not against zero — and as it turns out, our model does not beat it. We report that rather than bury it.

## 5. Model 1: TF-IDF + logistic regression

TF-IDF turns each tweet into numbers based on the words it contains, and logistic regression learns which words point towards a rise. This is deliberately the same first model the sample submission uses — a straightforward, inspectable baseline for text.

In [6]:
# 5 · Model 1 — exactly the approach the sample submission takes first.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vec = TfidfVectorizer()
Xtr = vec.fit_transform(X_train)
Xte = vec.transform(X_test)

model1 = LogisticRegression(max_iter=1000)
model1.fit(Xtr, y_train)
pred1 = model1.predict(Xte)

report("Model 1: TF-IDF + logistic regression", y_test, pred1)
print(confusion_matrix(y_test, pred1))
print(f"\n{len(vec.vocabulary_):,} distinct words and word-pairs learned from the training tweets.")

Model 1: TF-IDF + logistic regression
  accuracy  49.8%
  precision 47.7%  (of 'up' calls, how many were really up)
  recall    43.4%  (of real up days, how many we caught)
[[73 58]
 [69 53]]

3,381 distinct words and word-pairs learned from the training tweets.


## 6. Where Model 1 goes wrong

These are test tweets it labelled incorrectly, followed by the words it leans on most in each direction.

In [7]:
# 6 · The mistakes, and what the model is actually keying on.
mistakes = [(t, y, p) for t, y, p in zip(X_test, y_test, pred1) if y != p]
print(f"Model 1 got {len(mistakes)} of {len(y_test):,} test tweets wrong. Five of them:")
for text, true, said in mistakes[:5]:
    print(f"[really {'UP  ' if true else 'down'}, it said {'UP  ' if said else 'down'}] "
          f"{' '.join(str(text).split())[:125]}")

words = np.array(vec.get_feature_names_out())
print("\nWords pushing Model 1 towards 'up':  ", list(words[np.argsort(model1.coef_[0])[-12:]][::-1]))
print("Words pushing Model 1 towards 'down':", list(words[np.argsort(model1.coef_[0])[:12]]))

Model 1 got 127 of 253 test tweets wrong. Five of them:
[really UP  , it said down] $doge pump will be glorius X payments coming out any moment Watch it
[really UP  , it said down] First chart $BONK weekly possible devotion in play
[really UP  , it said down] $BONK - Looks like it needs a bit more time, back above blue and ill be loading up 🎯
[really down, it said UP  ] Telling you $DOGE season about to start 🫡 $LAIKA first layer 2 on doge also starting to run
[really down, it said UP  ] of all the major memecoins, $pepe looks the best imo. has already cleared and reclaimed the (a) wave lows after turbo V rever

Words pushing Model 1 towards 'up':   ['good', 'bottom', 'memecoins', 'me', 'keep', 'big', 'wif', 'giga', 'breakout', 'dip', 'see', 'dollar']
Words pushing Model 1 towards 'down': ['it', 'fresh', 'longed', 'move', 'weeks', 'there', 'memecoin', 'highest', 'still', '000', 'way', 'down']


Read those two lists carefully, because they are the most revealing output in the notebook.

The strongest "up" words are a mixture: a few genuinely directional ones (`breakout`, `dip`, `bottom`), the name of a coin (`wif`), and a lot of ordinary filler (`me`, `see`, `keep`, `good`, `dollar`). The "down" list is worse — `it`, `there`, `still`, `weeks`, `way`, `000`. This is mostly authors' writing habits rather than statements about tomorrow.

But the decisive evidence is not the word list, it is the score. **Model 1 is worse than the baseline that understands nothing**: 49.8% against 51.8%. Adding a text model to this problem made it *less* accurate, which means whatever it learned from the words is noise relative to the base rate.

That is worth testing directly on markets it has never seen, which is what the next section does.

## 7. Our addition: test it on pump.fun-era memecoins it has never seen

**This is the part we added beyond the brief, and it mirrors the sample submission's most important move** — the sample trained on old spam texts and then tested on scams people receive today. We do the same thing with markets.

We take four Solana memecoins from the pump.fun era — **WIF, ACT, PNUT and BOME**, all launched after late 2023 — and we test on every tweet about them. **Model 1 has never seen a single one of these tweets, and none of these four coins was in its training data.**

In [8]:
# 7 · The unseen-market test.
ood_pred = model1.predict(vec.transform(new.text))
new = new.assign(pred=ood_pred)

report(f"Model 1 on pump.fun-era memecoins ({len(new):,} tweets it never saw)", new.up, new.pred)
print(f"  for comparison, the base rate on this group is {max(new.up.mean(), 1 - new.up.mean()):.1%}")

print("\nPer coin:")
for coin in NEW:
    c = new.loc[new.coin.eq(coin)]
    if len(c):
        print(f"  {coin:6s} {len(c):4,d} tweets   up rate {c.up.mean():5.1%}   "
              f"model accuracy {accuracy_score(c.up, c.pred):5.1%}   ({'better' if accuracy_score(c.up, c.pred) > max(c.up.mean(), 1-c.up.mean()) else 'no better'} than always guessing)")

Model 1 on pump.fun-era memecoins (274 tweets it never saw)
  accuracy  48.9%
  precision 46.0%  (of 'up' calls, how many were really up)
  recall    73.4%  (of real up days, how many we caught)
  for comparison, the base rate on this group is 54.7%

Per coin:
  WIF     164 tweets   up rate 43.3%   model accuracy 45.7%   (no better than always guessing)
  ACT      61 tweets   up rate 44.3%   model accuracy 50.8%   (no better than always guessing)
  PNUT     33 tweets   up rate 63.6%   model accuracy 60.6%   (no better than always guessing)
  BOME     16 tweets   up rate 31.2%   model accuracy 50.0%   (no better than always guessing)


**This is our most important finding.** Model 1 scores 49.8% on the test set it was drawn from, and **48.9%** on memecoins from the launchpad era — against a base rate of 54.7%.

The explanation is the same one the sample submission reaches with modern scams. The model learned the vocabulary and habits of a handful of large-cap memecoin accounts. A pump.fun-era token is written about by different people, in a different register, at a different point in the market cycle. Nothing it learned transfers, because what it learned was never about price at all.

Note the shape of the failure: recall is high (73.4%) and precision is low (46.0%). The model calls "up" far more often than the data justifies, so it is over-eager rather than accurate. **All four coins come out no better than always guessing**, and on PNUT it scores 60.6% against a 63.6% base rate — beaten by the do-nothing rule even where it looks strongest.

## 8. Two honest checks on the 80/20 split

The brief asks for an 80/20 split, and section 5 gave us one. But a *random* split on this data has a specific problem we should measure rather than assume: tweets about the same coin on the same day share fate, so if one lands in training and its twin lands in test, the model can appear to predict the future when it is really recognising a day it has already seen.

We test that two ways.

In [9]:
# 8a · How leaky is the random split?
train_days = set(big.loc[X_train.index, "trading_day"])
test_days  = big.loc[X_test.index, "trading_day"]
overlap    = test_days.isin(train_days)
print(f"{overlap.mean():.1%} of the test tweets predict a trading day that ALSO appears in training.")
print("On those rows the model has already seen the answer for that coin-day.")

# 8b · The same model, split by time instead of at random.
ordered = big.sort_values("available_at").reset_index(drop=True)
cut = ordered.available_at.iloc[int(len(ordered) * 0.8)]
past, future = ordered.loc[ordered.available_at <= cut], ordered.loc[ordered.available_at > cut]
vec_t = TfidfVectorizer()
model_t = LogisticRegression(max_iter=1000)
model_t.fit(vec_t.fit_transform(past.text), past.up)
report(f"Chronological 80/20 — train to {str(cut)[:10]}, test after ({len(future):,} tweets)",
       future.up, model_t.predict(vec_t.transform(future.text)))
print(f"  base rate on the future portion: {max(future.up.mean(), 1 - future.up.mean()):.1%}")

# 8c · Making the model less lopsided, as the sample submission does.
model_bal = LogisticRegression(max_iter=1000, class_weight="balanced")
model_bal.fit(Xtr, y_train)
report("Model 3: TF-IDF + balanced class weights", y_test, model_bal.predict(Xte))

80.2% of the test tweets predict a trading day that ALSO appears in training.
On those rows the model has already seen the answer for that coin-day.
Chronological 80/20 — train to 2025-09-13, test after (252 tweets)
  accuracy  45.6%
  precision 40.7%  (of 'up' calls, how many were really up)
  recall    43.9%  (of real up days, how many we caught)
  base rate on the future portion: 54.8%
Model 3: TF-IDF + balanced class weights
  accuracy  49.0%
  precision 47.2%  (of 'up' calls, how many were really up)
  recall    49.2%  (of real up days, how many we caught)


**The honest split tells a different story from the flattering one.** On the random 80/20 split the model looks respectable; when we train only on the past and test only on what came after, the advantage largely disappears and lands within a couple of points of the base rate (45.6% against 54.8%). Note also that 80.2% of the random split's test rows predict a trading day that also sits in training — a real, measurable overlap.

The balanced-weights change in 8c does what it did in the sample submission: it trades accuracy for recall, which is the right trade if you care about catching ups, and the wrong one if you only look at accuracy. We report both rather than picking the flattering one.

## 9. The demo: HypeCheck explains its call

A model that just says "up" or "down" teaches a beginner nothing. So HypeCheck also says **which words in the tweet drove the answer**, using a hype lexicon we curated and then *measured* against this corpus rather than trusting.

Each term in `data/hype-lexicon.csv` carries a **hype score from 0 to 100**: the share of tweets containing that term which also carry an independent hype marker (an exclamation mark, heavy capitalisation, urgency, or an extreme claim), against a corpus baseline of 16%. Terms that the data contradicted were dropped, and terms that never appear are kept in the file flagged as absent, because their absence is a finding in itself.

In [10]:
# 9 · A lexicon we measured, not one we assumed.
lex = pd.read_csv("data/hype-lexicon.csv")

def terms_for(kind, statuses):
    sel = np.logical_and(lex.curated_as.eq(kind).values, lex.status.isin(statuses).values)
    return set(lex.loc[sel, "term"].str.lower())

BULL = terms_for("bullish", ["validated", "weak"])
BEAR = terms_for("bearish", ["validated", "weak"])
hype_rows = lex.loc[np.logical_and(lex.curated_as.eq("hype").values, lex.hype_meter.notna().values)]
HYPE = {str(t).lower(): float(m) for t, m in zip(hype_rows.term, hype_rows.hype_meter)}

print(f"Measured lexicon: {len(BULL)} bullish terms, {len(BEAR)} bearish terms, {len(HYPE)} hype terms.")
print(f"Of {len(lex)} curated terms, {(lex.status == 'absent').sum()} never appear in this corpus at all "
      f"— including {', '.join(lex.loc[lex.status.eq('absent'), 'term'].head(5).tolist())} ...")

Measured lexicon: 31 bullish terms, 2 bearish terms, 53 hype terms.
Of 234 curated terms, 67 never appear in this corpus at all — including to the moon, rip higher, rallying, uptrend, holding support ...


In [11]:
# 9b · HypeCheck's explanation.
def hype_check(tweet, model=model1, vectorizer=vec):
    low = tweet.lower()
    bull = sorted([t for t in BULL if t in low])
    bear = sorted([t for t in BEAR if t in low])
    hyp  = sorted([t for t in HYPE if t in low], key=lambda t: -HYPE[t])
    hype_score = int(np.mean([HYPE[t] for t in hyp])) if hyp else 0

    prob = model.predict_proba(vectorizer.transform([tweet]))[0, 1]
    call = "UP" if prob >= 0.5 else "DOWN"

    parts = []
    if bull: parts.append("bullish wording " + ", ".join(f'"{w}"' for w in bull[:3]))
    if bear: parts.append("bearish wording " + ", ".join(f'"{w}"' for w in bear[:3]))
    if hyp:  parts.append("hype wording " + ", ".join(f'"{w}"' for w in hyp[:3]))
    reasons = "; ".join(parts) if parts else "no words from our measured lexicon"

    return (f"Tomorrow: {call}  (model says {prob:.0%} chance of up)\n"
            f"   hype score {hype_score}/100   detected {reasons}")

for tweet in [
    "Short term we probably get more chop but memes with brand name recognition are putting in cycle bottoms. Bullish on $WIF.",
    "This is going to send it. Massive breakout, huge volume, rocket fuel from here. Dont miss it. $PEPE",
    "Boring day. No volume, price sitting below the range, I am watching and waiting. $DOGE",
]:
    print(" ".join(tweet.split())[:105] + " ...")
    print("  ", hype_check(tweet), "\n")

Short term we probably get more chop but memes with brand name recognition are putting in cycle bottoms.  ...
   Tomorrow: DOWN  (model says 49% chance of up)
   hype score 0/100   detected bullish wording "bottoms", "bullish" 

This is going to send it. Massive breakout, huge volume, rocket fuel from here. Dont miss it. $PEPE ...
   Tomorrow: DOWN  (model says 46% chance of up)
   hype score 41/100   detected bullish wording "breakout", "send", "send it"; hype wording "dont miss", "huge", "massive" 

Boring day. No volume, price sitting below the range, I am watching and waiting. $DOGE ...
   Tomorrow: DOWN  (model says 46% chance of up)
   hype score 38/100   detected hype wording "watch" 



Notice what the explanation reveals. The hypey tweet scores **41/100** and the demo names exactly why: `massive`, `huge` and `dont miss`, on top of validated directional words like `breakout` and `send it`. The flat, boring tweet still scores **38/100**, because a single measured hype term — `watch` — is present. That is a useful demonstration that a lexicon is a blunt instrument, and we would rather show it than hide it. Words people expect to matter ("to the moon", "100x", "rugpull", "rallying", "uptrend") are among the 67 curated terms that **never appear in this dataset once**.

That is the finding underneath the whole project: this corpus is technical-analysis commentary, not retail shilling. We are not sampling hype accounts, which is one more reason a hype-based price model does not work here.

## 10. Results summary

| | Random 80/20 test | Chronological 80/20 test | Unseen pump.fun-era coins |
|---|---|---|---|
| Base rate (majority class) | 51.8% | 54.8% | 54.7% |
| **Model 1: TF-IDF + logistic regression** | 49.8% | 45.6% | 48.9% |
| Model 3: + balanced class weights | 49.0% | — | — |

The word lists in section 6 explain all three columns. The model learned *who writes about which coin*, not *what the market will do*. It looks passable on the split it was drawn from, weakens when the split respects time, and is no better than a coin toss on markets from the launchpad era.

## 11. Limitations and next steps

- **The sample is small.** 1,538 labelled tweets across nine coins, and one coin (DOGE) supplies most of them. Confidence intervals on every number above are wide, and we have not quoted any as precise.
- **The target is noisy by construction.** A daily up/down move in a memecoin is close to a coin flip. Our earlier work on this project measured how large an edge you would need to detect one reliably at this sample size, and it is larger than anything we found.
- **Same-day tweets share a label.** Tweets about the same coin on the same day all point at the same outcome, so accuracy is not independent across rows. This is why the chronological split in section 8 matters more than the random one.
- **We can only use coins that have a Binance series.** Plenty of pump.fun-era tokens — POPCAT, FARTCOIN, GOAT, FWOG, MOODENG — have tweets in this corpus but no Binance daily bar, so they cannot be scored against our target at all. That is a data-availability limit, not a choice.
- **Next, we would:** label a few hundred tweets by hand for *intent* (directional call, promotion, news) rather than trusting the archive's ticker tags; widen the coin universe to the tokens that do have price history; and report a cost-sensitive measure, because a missed move and a false alarm cost a beginner very differently.

## 12. Sources and credits

- **Tweets:** [StephanAkkerman/financial-tweets-crypto](https://huggingface.co/datasets/StephanAkkerman/financial-tweets-crypto) (MIT).
- **Prices:** Binance spot daily klines (USDT pairs), public market data, pinned in our [repository](https://github.com/adeviwon/hypecheck).
- **Hype lexicon:** curated and measured by us; published as `data/hype-lexicon.csv`.
- **Libraries:** scikit-learn (TfidfVectorizer, LogisticRegression), pandas, NumPy, Requests.
- **AI tools:** we used an AI coding assistant to help debug pandas joins and to sanity-check the leakage measurement in section 8, and we read the scikit-learn documentation on `class_weight` before using it.
- The sample submission provided by Veritas AI shaped this notebook's structure, and we say so plainly: the 80/20 split, the majority-class baseline, the first TF-IDF model, the mistake analysis, the unseen-data test and the explanation demo are all modelled on it.

*Prototype, not financial advice. The results are the ones we actually measured, including the negative one.*